# LAB 05

In [1]:
import pygame
import math
import numpy as np

LARGURA, ALTURA = 900, 650
FPS = 60

def ajustar_angulo(angulo):
    return (angulo + math.pi) % (2 * math.pi) - math.pi

class Robo:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)
        self.theta = 0.0

        self.angulos = np.radians([-60, -30, 0, 30, 60])
        self.alcance = 160.0
        self.leituras = [self.alcance] * 5

        self.alvo = None
        self.distancia_desvio = 55.0
        self.distancia_parada = 20.0
        self.distancia_rodas = 34.0

    def medir(self, obstaculos):
        leituras = []

        for beta in self.angulos:
            angulo = self.theta + beta
            distancia = self.alcance

            for passo in range(2, int(self.alcance) + 1, 2):
                rx = self.x + passo * math.cos(angulo)
                ry = self.y + passo * math.sin(angulo)

                if rx <= 5 or rx >= LARGURA - 5 or ry <= 5 or ry >= ALTURA - 5:
                    distancia = float(passo)
                    break

                if any(obstaculo.collidepoint(rx, ry) for obstaculo in obstaculos):
                    distancia = float(passo)
                    break

            leituras.append(distancia)

        self.leituras = leituras

    def controlar(self):
        if self.alvo is None:
            return 0, 0, "Sem alvo"

        gx, gy = self.alvo

        dx = gx - self.x
        dy = gy - self.y

        distancia = math.hypot(dx, dy)

        if distancia < self.distancia_parada:
            return 0, 0, "Alvo atingido"

        angulo_alvo = math.atan2(dy, dx)
        erro = ajustar_angulo(angulo_alvo - self.theta)

        v = min(100, 0.9 * distancia)
        omega = 2.2 * erro
        modo = "Indo ao alvo"

        if min(self.leituras) < self.distancia_desvio:
            modo = "Desviando"

            torque = 0

            for beta, leitura in zip(self.angulos, self.leituras):
                if leitura < self.distancia_desvio:
                    intensidade = (self.distancia_desvio - leitura) / self.distancia_desvio
                    torque += -math.sin(beta) * intensidade

            if abs(torque) < 0.05:
                esquerda = self.leituras[0] + self.leituras[1]
                direita = self.leituras[3] + self.leituras[4]

                torque = -1 if esquerda < direita else 1

            omega = 3.5 * torque
            v = 35

        vl = v - omega * self.distancia_rodas / 2
        vr = v + omega * self.distancia_rodas / 2

        return vl, vr, modo

    def atualizar(self, dt, vl, vr):
        v = (vl + vr) / 2
        omega = (vr - vl) / self.distancia_rodas

        self.theta += omega * dt

        self.x += v * math.cos(self.theta) * dt
        self.y += v * math.sin(self.theta) * dt

        self.x = float(np.clip(self.x, 15, LARGURA - 15))
        self.y = float(np.clip(self.y, 15, ALTURA - 15))

    def desenhar(self, tela, fonte, modo):
        pygame.draw.circle(tela, (0, 200, 255), (int(self.x), int(self.y)), 14)

        for beta, distancia in zip(self.angulos, self.leituras):
            angulo = self.theta + beta

            ex = self.x + distancia * math.cos(angulo)
            ey = self.y + distancia * math.sin(angulo)

            pygame.draw.line(tela, (255, 210, 0), (self.x, self.y), (ex, ey), 1)

        if self.alvo:
            pygame.draw.circle(tela, (80, 255, 120), self.alvo, 8, 2)

        tela.blit(fonte.render(f"Modo: {modo}", True, (240, 240, 240)), (20, 20))

def main():
    pygame.init()

    tela = pygame.display.set_mode((LARGURA, ALTURA))
    pygame.display.set_caption("LAB 05 - Go to Goal")
    relogio = pygame.time.Clock()
    fonte = pygame.font.SysFont("monospace", 14)

    robo = Robo(100, 550)

    obstaculos = [
        pygame.Rect(250, 100, 100, 330),
        pygame.Rect(470, 300, 150, 100),
        pygame.Rect(690, 120, 90, 300)
    ]

    rodando = True

    while rodando:
        dt = relogio.tick(FPS) / 1000

        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                rodando = False

            if evento.type == pygame.MOUSEBUTTONDOWN and evento.button == 1:
                robo.alvo = evento.pos

        robo.medir(obstaculos)

        vl, vr, modo = robo.controlar()

        robo.atualizar(dt, vl, vr)

        tela.fill((20, 24, 30))

        for obstaculo in obstaculos:
            pygame.draw.rect(tela, (180, 50, 50), obstaculo)

        robo.desenhar(tela, fonte, modo)

        pygame.display.flip()

    pygame.quit()

main()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
